# Notebook 02: Pemrosesan Dataset Wajah Aktor, Pelatihan FaceNet Triplet Loss, & Evaluasi Matrix Score

Notebook ini menjalankan **Fokus 2**, yaitu memproses dataset citra wajah aktor dengan memanggil hasil akhir model detektor wajah `storage/app/ai/weights/hasil_training_yolov8_wajah.pt` (dari **Notebook 01**), kemudian:
1. **Tahap 01:** Memotong kotak wajah aktor & orang random dengan margin `12%`, melakukan augmentasi pose/cahaya, serta menormalisasi piksel ke ukuran **`160×160` RGB**.
2. **Tahap 02:** Melatih arsitektur **`JaringanFaceNetMandiri` (`torch.nn.Module`)** dari **Epoch 0** menggunakan fungsi **Triplet Margin Loss (`Anchor, Positive, Negative`)** di **GPU NVIDIA GeForce RTX 4060** hingga menghasilkan `storage/app/ai/weights/hasil_training_facenet_aktor.pth`.
3. **Tahap 03:** Mengekstrak vektor referensi **`512-D` (`.npy`)** ke folder `python/embeddings/`.
4. **Tahap 04:** Menghitung **Euclidean Distance**, mengkalibrasi *threshold* optimal, dan menghasilkan **Confusion Matrix (`TP, TN, FP, FN`)**, **Accuracy**, **Precision**, **Recall**, serta **F1-Score**.

In [ ]:
# Mengatur path absolut root proyek agar seluruh modul terbaca dengan benar dari Jupyter Notebook lokal
import sys
from pathlib import Path

FOLDER_NOTEBOOK_INI = Path.cwd().resolve()
if FOLDER_NOTEBOOK_INI.name == "02_training_dataset_wajah_aktor":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent.parent
elif FOLDER_NOTEBOOK_INI.name == "training_python":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent
else:
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI

for path_modul in [
    FOLDER_ROOT_PROYEK,
    FOLDER_ROOT_PROYEK / "training_python",
    FOLDER_ROOT_PROYEK / "training_python" / "02_training_dataset_wajah_aktor",
    FOLDER_ROOT_PROYEK / "python",
]:
    if str(path_modul) not in sys.path:
        sys.path.insert(0, str(path_modul))

import konfigurasi_training as konfig
from facenet.arsitektur_facenet_manual import JaringanFaceNetMandiri
from tahap_01_crop_dan_preprocess_wajah import ekstrak_dan_preprocess_wajah_dataset
from tahap_02_training_facenet_triplet import latih_model_facenet_triplet_loss
from tahap_03_ekstraksi_embedding_aktor import buat_database_embedding_referensi
from tahap_04_evaluasi_euclidean_dan_f1 import kalibrasi_threshold_dan_evaluasi_matrix_f1

# Memverifikasi penguncian GPU NVIDIA GeForce RTX 4060 Laptop GPU dan memeriksa bobot YOLOv8 dari Notebook 01
indeks_cuda, objek_device_gpu, info_perangkat_gpu = konfig.pilih_perangkat_gpu_rtx_wajib(wajib_cuda=True)
jaringan_uji = JaringanFaceNetMandiri(dimensi_embedding=konfig.DIMENSI_VEKTOR_EMBEDDING_FACENET)

print("Root Proyek             :", FOLDER_ROOT_PROYEK)
print("Perangkat GPU Aktif     :", info_perangkat_gpu["nama_gpu"], f"({info_perangkat_gpu['vram_gb']} GB VRAM)")
print("Indeks CUDA             :", info_perangkat_gpu["device_pytorch"], "-", info_perangkat_gpu["catatan_task_manager"])
print("Bobot YOLOv8 Tersedia   :", konfig.FILE_BOBOT_HASIL_TRAINING_YOLO.exists(), "->", konfig.FILE_BOBOT_HASIL_TRAINING_YOLO)
print("Jumlah Parameter FaceNet:", sum(p.numel() for p in jaringan_uji.parameters()))

## Tahap 01: Pemotongan Kotak Wajah Aktor (Margin 12%) & Preprocessing `160×160` RGB
Memanggil `hasil_training_yolov8_wajah.pt` untuk mendeteksi wajah pada foto aktor (dari database `/camp` maupun `wajah_aktor_target/`) dan `wajah_orang_random/`, menambahkan margin tepi `12%`, menyekalakan ke `160×160` RGB, membuat variasi augmentasi pose/pencahayaan, serta menyimpan tensor ternormalisasi ke `crop_wajah_160x160/`.

In [ ]:
# Menjalankan deteksi crop wajah dan preprocessing 160x160 RGB beserta augmentasi data latih
ringkasan_preprocessing = ekstrak_dan_preprocess_wajah_dataset(
    aktifkan_augmentasi_aktor=True,
    rasio_margin_crop=konfig.RASIO_MARGIN_CROP_WAJAH,
)

## Tahap 02: Pelatihan Mandiri `JaringanFaceNetMandiri` dengan Triplet Margin Loss (Epoch 0)
Menyusun pasangan `(Anchor, Positive, Negative)` dan melatih jaringan saraf FaceNet mandiri dari Epoch 0 di GPU RTX 4060 hingga konvergen (dilengkapi *Early Stopping*), lalu menyimpan bobot terbaik ke `storage/app/ai/weights/hasil_training_facenet_aktor.pth`.

In [ ]:
# Melatih JaringanFaceNetMandiri menggunakan fungsi Triplet Margin Loss di GPU RTX 4060
data_hasil_training_facenet = latih_model_facenet_triplet_loss(
    jumlah_epoch=konfig.JUMLAH_EPOCH_TRAINING_FACENET,
    batas_patience=konfig.BATAS_PATIENCE_EARLY_STOPPING_FACENET,
    ukuran_batch=konfig.UKURAN_BATCH_TRIPLET_FACENET,
    laju_belajar_awal=konfig.LAJU_BELAJAR_AWAL_FACENET,
    batas_margin_triplet=konfig.BATAS_MARGIN_TRIPLET_LOSS,
)

## Tahap 03: Pembuatan Database Reference Embedding `512-D` (`.npy`)
Mengekstrak vektor embedding `512-D` (`L2-normalized`) untuk setiap foto asli aktor memakai `hasil_training_facenet_aktor.pth` dan menyimpannya ke `python/embeddings/{slug_aktor}/{image_id}.npy` beserta `manifest.json`.

In [ ]:
# Mengekstrak dan menyimpan vektor reference embedding .npy 512-D ke folder python/embeddings/
ringkasan_ekstraksi_embedding = buat_database_embedding_referensi()

## Tahap 04: Evaluasi Euclidean Distance, Kalibrasi Threshold, Confusion Matrix, & F1-Score
Menghitung distribusi jarak Euclidean *intra-class* vs *inter-class*, mencari nilai `batas_ambang_jarak` (*threshold*) optimal, serta menghasilkan tabel **Confusion Matrix (`TP, TN, FP, FN`)** dan metrik **Accuracy, Precision, Recall, serta F1-Score**.

In [ ]:
# Menjalankan kalibrasi threshold Euclidean Distance dan evaluasi Confusion Matrix serta F1-Score
laporan_evaluasi_lengkap = kalibrasi_threshold_dan_evaluasi_matrix_f1()